# Exploratory Data Analysis (EDA) — Kyivan Dataset

Этот ноутбук предназначен для анализа финального собранного датасета исторических славянских текстов.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import numpy as np
from collections import Counter

# Настройки графиков
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Загрузка данных
Читаем `final_dataset.jsonl` в `pandas.DataFrame`.


In [ ]:
data = []
with open('../data/final_dataset.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)
df['text_len_chars'] = df['text'].apply(len)
df['text_len_words'] = df['text'].apply(lambda x: len(x.split()))

print(f"Всего документов: {len(df)}")
display(df.head())


## 2. Распределение Макро-Диалектов
Посмотрим, как сбалансированы наши 4 класса: `OES` (древнерусский), `CS` (церковнославянский), `NW` (новгородский) и `SW` (западнорусский).


In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='macro_dialect', order=df['macro_dialect'].value_counts().index)
plt.title('Распределение текстов по диалектам')
plt.xlabel('Macro Dialect')
plt.ylabel('Количество текстов')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + 0.3, p.get_height() + 50))
plt.show()


## 3. Распределение по времени (Вектора дат)
У нас есть 20-мерные вектора `date_target`, представляющие корзины по 50 лет от 800 до 1800 года.
Суммируем все вектора, чтобы получить общее распределение текстов во времени.


In [ ]:
# Извлекаем все вектора и суммируем их (каждый вектор - это вероятность)
all_targets = np.array(df['date_target'].tolist())
sum_targets = all_targets.sum(axis=0)

# Подготовим подписи для корзин
start_year = 800
bucket_size = 50
labels = [f"{start_year + i*bucket_size}-{start_year + (i+1)*bucket_size - 1}" for i in range(20)]

plt.figure(figsize=(14, 6))
sns.barplot(x=labels, y=sum_targets, color='royalblue')
plt.xticks(rotation=45)
plt.title('Распределение текстов по времени (Сумма вероятностей из date_target)')
plt.xlabel('Исторический период')
plt.ylabel('Ожидаемое количество документов')
plt.tight_layout()
plt.show()


## 4. Длина текстов
Проанализируем среднюю длину текста в зависимости от диалекта/источника.


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='macro_dialect', y='text_len_words')
plt.yscale('log')
plt.title('Распределение длины текстов (в словах) по диалектам (Log Scale)')
plt.ylabel('Количество слов')
plt.show()


## 5. Анализ символов (Специфика исторической орфографии)
Посмотрим на самые частые и самые редкие символы в нашем корпусе, чтобы убедиться, что мы сохранили «юсы» и «омеги».


In [ ]:
all_text = " ".join(df['text'].tolist())
char_counts = Counter(all_text)

# Исключаем пробелы и стандартные знаки препинания для наглядности
import string
exclude = set(string.ascii_letters + string.punctuation + string.digits + " 

")
filtered_chars = {k: v for k, v in char_counts.items() if k not in exclude}

top_chars = dict(Counter(filtered_chars).most_common(20))
bottom_chars = dict(Counter(filtered_chars).most_common()[-30:])

print("--- ТОП 20 кириллических символов ---")
print(top_chars)

print("\n--- Редкие символы (Юсы, Омеги, диакритика) ---")
print(bottom_chars)
